In [ ]:
# ============================
# ADMET-AI Topical Screening
# ============================

# Install/load packages if needed
req_pkgs <- c("readr", "dplyr", "stringr", "janitor", "tidyr", "purrr")
to_install <- setdiff(req_pkgs, rownames(installed.packages()))
if (length(to_install)) install.packages(to_install)
lapply(req_pkgs, library, character.only = TRUE)

# --------- CONFIGURE THRESHOLDS (tweak as needed) ---------
# Strict filters
THRESH_LOGKP_MAX <- -5.0         # Prefer < -5.0 cm/s (if column available)
TPSA_MIN <- 20                   # Å^2
TPSA_MAX <- 120
ALLOW_BBB <- FALSE               # BBB permeation disallowed for topical
CYP_BLOCK_THRESHOLD <- 0.5       # Prob > 0.5 = likely inhibitor/substrate (flags risk)
SOL_LOGS_MIN <- -4               # Solubility_AqSolDB (logS). Higher is more soluble.

# Relaxable filters (allow 1-2% topical, formulation tricks)
MW_MAX <- 500
LOGP_MAX <- 5.5
LIPINSKI_VIOL_MAX <- 1           # <= 1 violation tolerated
SA_MAX <- 6                      # Synthetic accessibility (if present)

# List of CYP columns to check (probabilities 0..1)
cyp_cols <- c(
  "cyp1a2_veith", "cyp2c19_veith", "cyp2c9_veith", "cyp2c9_substrate_carbonmangels",
  "cyp2d6_veith", "cyp2d6_substrate_carbonmangels",
  "cyp3a4_veith", "cyp3a4_substrate_carbonmangels"
)

# --------- LOAD DATA ----------
# Replace with your actual file path:
admet <- readr::read_csv("admet_ai_export.csv") %>%
  janitor::clean_names()   # make names snake_case

# Peek
# names(admet)

# Helper to safely get column or fill NA if missing
get_col <- function(df, name) if (name %in% names(df)) df[[name]] else rep(NA, nrow(df))

# Try to infer Lipinski violations if not present
infer_lipinski <- function(mw, logp, hba, hbd) {
  # <= 1 violation preferred in relaxable filters
  # Rule of 5: MW <= 500, logP <= 5, HBA <= 10, HBD <= 5
  v <- (mw > 500) + (logp > 5) + (hba > 10) + (hbd > 5)
  as.integer(v)
}

# Add normalized screening columns
df <- admet %>%
  mutate(
    compound = coalesce(get_col(., "phytochemical"), get_col(., "name"), get_col(., "molecule"), row_number()),
    mw       = get_col(., "molecular_weight"),
    logp     = coalesce(get_col(., "logp"), get_col(., "consensus_logp")),  # sometimes named differently
    tpsa     = get_col(., "tpsa"),

    # If LogKp exists (rare in ADMET-AI export), use it; else NA
    logkp    = get_col(., "log_kp"),

    # Lipinski violations (if not provided, infer)
    lipinski_vio = coalesce(get_col(., "lipinski"), infer_lipinski(mw, logp,
                                                                   get_col(., "hydrogen_bond_acceptors"),
                                                                   get_col(., "hydrogen_bond_donors"))),

    # Solubility: use class if available, else numeric logS threshold
    sol_class = get_col(., "solubility_class"),
    sol_logs  = get_col(., "solubility_aqsoldb"),

    # BBB & toxicity surrogates
    bbb      = get_col(., "bbb_martins"),   # 0..1; >0.5 considered permeant
    skin_rxn = get_col(., "skin_reaction"),
    clintox  = get_col(., "clintox"),
    dili     = get_col(., "dili"),

    # CYP risks (probabilities)
    across(all_of(cyp_cols), ~ get_col(., cur_column()))
  )

# Build strict filter flags
df <- df %>%
  mutate(
    # Strict solubility OK if class says Soluble/Moderately soluble OR logS > -4
    sol_ok = case_when(
      !is.na(sol_class) ~ str_detect(tolower(sol_class), "soluble|moderately"),
      !is.na(sol_logs)  ~ sol_logs > SOL_LOGS_MIN,
      TRUE ~ NA
    ),

    tpsa_ok = tpsa >= TPSA_MIN & tpsa <= TPSA_MAX,
    logkp_ok = ifelse(is.na(logkp), NA, logkp < THRESH_LOGKP_MAX),
    bbb_ok = ifelse(is.na(bbb), NA, ifelse(ALLOW_BBB, TRUE, bbb <= 0.5)),
    # Any CYP risk above threshold flags a risk
    cyp_risk = pmap_lgl(select(., all_of(cyp_cols)), ~ any(c(...) > CYP_BLOCK_THRESHOLD, na.rm = TRUE)),

    strict_ok = coalesce(sol_ok, TRUE) &
                coalesce(tpsa_ok, TRUE) &
                coalesce(bbb_ok, TRUE) &
                # If LogKp missing, we don't fail it; else require pass
                coalesce(logkp_ok, TRUE) &
                # We want minimal CYP risk under strict mode
                !coalesce(cyp_risk, FALSE)
  )

# Build relaxable filter flags
df <- df %>%
  mutate(
    mw_ok_rel    = coalesce(mw < MW_MAX, TRUE),
    logp_ok_rel  = coalesce(logp < LOGP_MAX, TRUE),
    lip_ok_rel   = coalesce(lipinski_vio <= LIPINSKI_VIOL_MAX, TRUE),
    # SA not in your export; if present, enforce; else ignore
    sa_ok_rel    = if ("synthetic_accessibility" %in% names(.)) get_col(., "synthetic_accessibility") < SA_MAX else TRUE,

    relax_ok     = mw_ok_rel & logp_ok_rel & lip_ok_rel & sa_ok_rel
  )

# Decide bins:
# - STRICT: strict_ok == TRUE
# - FORMULATION-DEPENDENT: strict_ok == FALSE AND relax_ok == TRUE
# - EXCLUDED: relax_ok == FALSE
df <- df %>%
  mutate(
    bucket = case_when(
      strict_ok ~ "STRICT",
      !strict_ok & relax_ok ~ "FORMULATION_DEPENDENT",
      TRUE ~ "EXCLUDED"
    )
  )

# Build reasons for exclusion/formulation-dependence
reason_cols <- c(
  "sol_ok","tpsa_ok","logkp_ok","bbb_ok","cyp_risk",
  "mw_ok_rel","logp_ok_rel","lip_ok_rel","sa_ok_rel"
)

df <- df %>%
  mutate(across(all_of(reason_cols), ~ ifelse(is.na(.x), NA, as.logical(.x)))) %>%
  rowwise() %>%
  mutate(
    reasons = {
      r <- c()
      if (!coalesce(sol_ok, TRUE))   r <- c(r, "solubility")
      if (!coalesce(tpsa_ok, TRUE))  r <- c(r, "TPSA")
      if (!is.na(logkp_ok) && !logkp_ok) r <- c(r, "LogKp")
      if (!coalesce(bbb_ok, TRUE))   r <- c(r, "BBB")
      if (coalesce(cyp_risk, FALSE)) r <- c(r, "CYP risk")

      if (!coalesce(mw_ok_rel, TRUE))   r <- c(r, "MW (relax)")
      if (!coalesce(logp_ok_rel, TRUE)) r <- c(r, "logP (relax)")
      if (!coalesce(lip_ok_rel, TRUE))  r <- c(r, "Lipinski (relax)")
      if (!coalesce(sa_ok_rel, TRUE))   r <- c(r, "SA (relax)")
      if (length(r) == 0) NA_character_ else paste(unique(r), collapse = "; ")
    }
  ) %>%
  ungroup()

# Export CSVs
strict_out <- df %>% filter(bucket == "STRICT") %>% arrange(compound)
formdep_out <- df %>% filter(bucket == "FORMULATION_DEPENDENT") %>% arrange(compound)
excluded_out <- df %>% filter(bucket == "EXCLUDED") %>% arrange(compound)

readr::write_csv(strict_out, "topical_strict.csv")
readr::write_csv(formdep_out, "topical_formulation_dep.csv")
readr::write_csv(excluded_out, "topical_excluded.csv")

# Console summary
cat("\n=== Topical Screening Summary ===\n")
cat("Strict candidates:            ", nrow(strict_out), "\n")
cat("Formulation-dependent:        ", nrow(formdep_out), "\n")
cat("Excluded:                     ", nrow(excluded_out), "\n\n")

cat("Top reasons for EXCLUSION/formulation-dependence:\n")
excluded_out %>%
  filter(!is.na(reasons)) %>%
  separate_rows(reasons, sep = ";\\s*") %>%
  count(reasons, sort = TRUE) %>%
  print(n = 20)

cat("\nOutputs written:\n - topical_strict.csv\n - topical_formulation_dep.csv\n - topical_excluded.csv\n")
